# Colab 引导：加载代码库与活跃训练线

本笔记本在 Colab 上完成：克隆/更新仓库 → 依赖安装（`install_env.py` 自动适配平台）→ 冒烟验证 → 启动当前活跃训练入口。

当前活跃入口（活跃/预留清单见 `knowledge/code_analysis.md` §五）：
- **§4 Crafter DreamerV3** — 世界模型线（采集 ↔ 世界模型更新 ↔ 想象 actor-critic），训练手册见 `knowledge/dreamer.md`。
- **§5 Crafter PPO + Achievement Distillation** — model-free 基线。

Craftground（Minecraft 1.21）域需要 Java 21 与真实游戏进程，安装用 `python install_env.py --craftground`；建议在独立 GPU 机器运行，不在本笔记本覆盖范围。

旧版 Δz-JEPA 训练/评估 demo（`train_minecraft.py`/`train_rssm.py`/`net/world_model.py` 等）已随退役线删除（commit `25e8355`），如需参考取 git 历史。DINOv3 特征预提取见 standalone 的 `transfer.ipynb`（不依赖本仓库）。

**安全提示**：密钥一律从 Colab Secrets（左侧 🔑）注入，不硬编码。私有仓库需要 `GH_PAT`（repo 读权限）；仓库公开时留空即可。

**产物**：checkpoints 与日志统一落在 `<REPO_DIR>/runs/`（gitignored），Colab 断线前记得另存到 Drive。

In [ ]:
# ===== §1 配置与获取代码库 =====
import os

REPO     = "OopsYouDiedE/tao-not-42-base-refactor-world-model-contract"
REPO_DIR = "/content/repo"

# GH_PAT 从 Colab Secrets(左侧 🔑)注入;非 Colab 环境回退到环境变量;公开仓库留空即可
try:
    from google.colab import userdata
    try:
        PAT = userdata.get("GH_PAT") or ""
    except Exception:
        PAT = ""
except ImportError:
    PAT = os.environ.get("GH_PAT", "")

_auth = f"{PAT}@" if PAT else ""
CLONE_URL = f"https://{_auth}github.com/{REPO}.git"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !rm -rf {REPO_DIR}
    !git clone --depth=1 {CLONE_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth=1 origin main && git reset --hard origin/main
!cd {REPO_DIR} && git log --oneline -1

In [ ]:
# ===== §2 安装依赖 =====
# install_env.py 自动适配平台(is_colab/is_headless 检测、uv 自装、editable 安装本包)。
# --dreamer + --ppo-ad 覆盖 §4/§5 两条活跃线;--dev 提供 pytest 供 §3 冒烟。
# 换 Craftground 域时改用: python install_env.py --craftground (Java 21 由脚本处理)。
!cd {REPO_DIR} && python install_env.py --dreamer --ppo-ad --dev

In [ ]:
# ===== §3 冒烟验证:代码库加载成功的判据 =====
# tests/unit(纯算子,CPU)+ tests/integration/test_dreamer_build(DI mock 骨干,
# DreamerV3 loss+反向+policy 与 Dreamer4 前向)。全部通过 = 加载与依赖安装成立。
!cd {REPO_DIR} && python -m pytest tests/unit tests/integration/test_dreamer_build.py -q

In [ ]:
# ===== §4 训练 · Crafter DreamerV3(世界模型线) =====
# 采集 ↔ 世界模型更新 ↔ 想象 actor-critic;超参与规模选择见 knowledge/dreamer.md。
# --size {tiny,small,default}:T4/L4 从 small 起步。
# 断点续训:加 --resume runs/crafter_dreamerv3/<ckpt>.pt。
!cd {REPO_DIR} && python train/crafter/train_dreamerv3.py \
  --size small \
  --total-steps 200000 \
  --run-dir runs/crafter_dreamerv3

In [ ]:
# ===== §5 训练 · Crafter PPO + Achievement Distillation(model-free 基线) =====
# 与 §4 独立使用(各自单独进程,别同时跑满同一块 GPU)。
# --vec subproc:多进程向量环境;Colab 单核受限时可退回 --vec serial。
!cd {REPO_DIR} && python train/crafter/train_ppo_ad.py \
  --total-timesteps 1000000 \
  --vec subproc \
  --run-dir runs/crafter_ppo_ad